<a href="https://colab.research.google.com/github/Ratish-bit/AI-Assisted-Threat-Detection-dashboard/blob/main/notebooks/SQL_Threat_Detection_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Task – AI-Assisted Threat Detection Dashboard
## Cybersecurity Threat Events Database Analysis

**Project:** AI-Assisted Threat Detection Dashboard
**Task:** Create a synthetic cybersecurity threat events dataset, load it into an in-memory SQL database using SQLite, and analyse it using various SQL clauses including WHERE, GROUP BY, HAVING, ORDER BY, BETWEEN, IN, and JOIN.

**Tools Used:** Python, SQLite3, Pandas

---
### Tasks Overview
| Task | Description | SQL Concepts |
|------|-------------|---------------|
| Task 1 | Setup database and create dataset | CREATE TABLE, INSERT |
| Task 2 | View all events | SELECT, LIMIT |
| Task 3 | Filter high-risk events | WHERE, ORDER BY |
| Task 4 | Count events by threat type | GROUP BY, COUNT, AVG |
| Task 5 | Top source IPs by attack volume | GROUP BY, HAVING |
| Task 6 | Filter by protocol and threat label | WHERE with IN, AND |
| Task 7 | Risk score range analysis | BETWEEN |
| Task 8 | Asset-level threat join | LEFT JOIN |
| Task 9 | Aggregated dashboard summary | GROUP BY, SUM, AVG |
| Task 10 | High severity recent events | WHERE, ORDER BY, LIMIT |

In [ ]:
# ============================================================
# TASK 1: Setup - Import Libraries & Create Synthetic Dataset
# Prompt: Import SQLite3 and Pandas. Create a synthetic
# cybersecurity threat events dataset with 1000 rows and
# load it into an in-memory SQLite database.
# ============================================================

import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
n = 1000

# Generate synthetic threat events dataset
start_date = datetime(2026, 1, 1)
timestamps = [start_date + timedelta(hours=np.random.randint(0, 4320)) for _ in range(n)]

source_ips  = [f"192.168.{np.random.randint(1,10)}.{np.random.randint(1,255)}" for _ in range(n)]
dest_ips    = [f"10.0.{np.random.randint(0,5)}.{np.random.randint(1,50)}"      for _ in range(n)]
protocols   = np.random.choice(["TCP","UDP","HTTP","HTTPS","SSH","ICMP"], n, p=[0.3,0.2,0.2,0.15,0.1,0.05])
labels      = np.random.choice(["Normal","BruteForce","DataExfiltration","MalwareBeacon","SuspiciousScan"],
                                n, p=[0.55,0.15,0.12,0.10,0.08])
asset_crit  = np.random.randint(1, 6, n)
bytes_sent  = np.where(labels == "DataExfiltration",
                        np.random.randint(50000, 200000, n),
                        np.random.randint(100, 15000, n))
bytes_recv  = np.random.randint(100, 10000, n)
pkt_count   = np.where(labels == "MalwareBeacon",
                        np.random.randint(5, 50, n),
                        np.random.randint(50, 2000, n))
failed_logins = np.where(labels == "BruteForce",
                          np.random.randint(10, 40, n),
                          np.random.randint(0, 3, n))

# Risk score formula: 0.4*severity + 0.3*asset_crit + 0.2*vuln + 0.1*anomaly
severity_map = {"Normal":2,"BruteForce":7,"DataExfiltration":9,"MalwareBeacon":8,"SuspiciousScan":5}
vuln_exp     = np.random.randint(1, 6, n)
anomaly_sc   = np.random.uniform(0, 10, n).round(2)

risk_score = np.array([
    round(0.4*severity_map[l] + 0.3*asset_crit[i] + 0.2*vuln_exp[i] + 0.1*anomaly_sc[i], 2)
    for i, l in enumerate(labels)
])

df = pd.DataFrame({
    "event_id"       : range(1, n+1),
    "timestamp"      : [t.strftime("%Y-%m-%d %H:%M:%S") for t in timestamps],
    "source_ip"      : source_ips,
    "destination_ip" : dest_ips,
    "protocol"       : protocols,
    "bytes_sent"     : bytes_sent,
    "bytes_received" : bytes_recv,
    "packet_count"   : pkt_count,
    "failed_logins"  : failed_logins,
    "asset_criticality" : asset_crit,
    "vulnerability_exposure" : vuln_exp,
    "anomaly_score"  : anomaly_sc,
    "risk_score"     : risk_score,
    "label"          : labels
})

# Load into SQLite in-memory database
conn = sqlite3.connect(":memory:")
df.to_sql("ThreatEvents", conn, index=False, if_exists="replace")

print(f"Dataset created: {len(df)} rows, {len(df.columns)} columns")
print(f"Label distribution:\n{df['label'].value_counts()}")
df.head()

Dataset created: 1000 rows, 14 columns
Label distribution:
label
Normal              573
BruteForce          132
DataExfiltration    118
MalwareBeacon       104
SuspiciousScan       73
Name: count, dtype: int64


,event_id,timestamp,source_ip,destination_ip,protocol,bytes_sent,bytes_received,packet_count,failed_logins,asset_criticality,vulnerability_exposure,anomaly_score,risk_score,label
0,1,2026-02-05 20:00:00,192.168.2.252,10.0.1.39,TCP,7494,6437,733,1,2,5,6.16,3.02,Normal
1,2,2026-06-07 04:00:00,192.168.6.107,10.0.3.5,TCP,6537,2629,1345,2,5,1,1.87,2.69,Normal
2,3,2026-05-09 20:00:00,192.168.1.51,10.0.0.27,HTTPS,4795,4070,1575,2,2,1,1.77,1.78,Normal
3,4,2026-01-20 10:00:00,192.168.5.36,10.0.4.47,TCP,11630,3507,1735,0,2,1,8.73,2.47,Normal
4,5,2026-05-24 12:00:00,192.168.3.26,10.0.0.47,TCP,14412,9750,1435,0,2,1,7.43,3.54,SuspiciousScan


In [ ]:
# ============================================================
# TASK 2: SELECT all events - VIEW FIRST 15 ROWS
# Prompt: Write a SQL query to select all columns from the
# ThreatEvents table and display the first 15 rows.
# SQL Concepts: SELECT, *, LIMIT
# ============================================================

query2 = """
SELECT *
FROM ThreatEvents
LIMIT 15;
"""

result2 = pd.read_sql(query2, conn)
print("Task 2: First 15 Threat Events")
result2

Task 2: First 15 Threat Events


,event_id,timestamp,source_ip,destination_ip,protocol,bytes_sent,bytes_received,packet_count,failed_logins,asset_criticality,vulnerability_exposure,anomaly_score,risk_score,label
0,1,2026-02-05 20:00:00,192.168.2.252,10.0.1.39,TCP,7494,6437,733,1,2,5,6.16,3.02,Normal
1,2,2026-06-07 04:00:00,192.168.6.107,10.0.3.5,TCP,6537,2629,1345,2,5,1,1.87,2.69,Normal
2,3,2026-05-09 20:00:00,192.168.1.51,10.0.0.27,HTTPS,4795,4070,1575,2,2,1,1.77,1.78,Normal
3,4,2026-01-20 10:00:00,192.168.5.36,10.0.4.47,TCP,11630,3507,1735,0,2,1,8.73,2.47,Normal
4,5,2026-05-24 12:00:00,192.168.3.26,10.0.0.47,TCP,14412,9750,1435,0,2,1,7.43,3.54,SuspiciousScan
5,6,2026-05-13 03:00:00,192.168.5.11,10.0.3.18,UDP,177602,9146,195,1,4,1,6.59,5.66,DataExfiltration
6,7,2026-05-02 15:00:00,192.168.8.11,10.0.1.2,TCP,176270,7516,992,0,2,2,4.07,5.01,DataExfiltration
7,8,2026-01-06 10:00:00,192.168.4.113,10.0.3.30,HTTP,1882,4860,1282,0,1,4,2.62,2.16,Normal
8,9,2026-03-12 05:00:00,192.168.7.232,10.0.0.36,TCP,6047,1997,32,1,1,4,9.79,5.28,MalwareBeacon
9,10,2026-02-02 01:00:00,192.168.3.54,10.0.1.37,SSH,6694,6464,1083,2,5,2,7.97,3.50,Normal


In [ ]:
# ============================================================
# TASK 3: WHERE Clause - Filter High Risk Events
# Prompt: Write a SQL query to retrieve all events where
# risk_score is greater than or equal to 7.0.
# Order results by risk_score descending.
# SQL Concepts: WHERE, ORDER BY, comparison operators
# ============================================================

query3 = """
SELECT event_id, timestamp, source_ip, destination_ip,
       protocol, label, risk_score
FROM ThreatEvents
WHERE risk_score >= 7.0
ORDER BY risk_score DESC;
"""

result3 = pd.read_sql(query3, conn)
print(f"Task 3: High-Risk Events (risk_score >= 7.0) -- {len(result3)} records found")
result3.head(15)

Task 3: High-Risk Events (risk_score >= 7.0) -- 1 records found


,event_id,timestamp,source_ip,destination_ip,protocol,label,risk_score
0,700,2026-01-01 03:00:00,192.168.5.54,10.0.2.44,TCP,DataExfiltration,7.06


In [ ]:
# ============================================================
# TASK 4: GROUP BY - Count Events by Threat Label
# Prompt: Write a SQL query to count total events per threat
# label and calculate the average risk score for each.
# SQL Concepts: GROUP BY, COUNT, AVG, ORDER BY
# ============================================================

query4 = """
SELECT label,
       COUNT(*)                    AS total_events,
       ROUND(AVG(risk_score), 2)   AS avg_risk_score,
       ROUND(AVG(bytes_sent), 0)   AS avg_bytes_sent,
       ROUND(AVG(failed_logins), 2) AS avg_failed_logins
FROM ThreatEvents
GROUP BY label
ORDER BY total_events DESC;
"""

result4 = pd.read_sql(query4, conn)
print("Task 4: Event Count and Average Risk Score by Threat Label")
result4

Task 4: Event Count and Average Risk Score by Threat Label


,label,total_events,avg_risk_score,avg_bytes_sent,avg_failed_logins
0,Normal,573,2.76,7144.0,1.03
1,BruteForce,132,4.82,7629.0,23.86
2,DataExfiltration,118,5.60,125813.0,1.01
3,MalwareBeacon,104,5.19,7151.0,1.04
4,SuspiciousScan,73,4.07,7826.0,1.18


In [ ]:
# ============================================================
# TASK 5: HAVING Clause - Top Source IPs by Attack Volume
# Prompt: Find source IPs that have generated more than 5
# high-risk events (risk_score >= 7). Show only those IPs
# with more than 5 such events.
# SQL Concepts: GROUP BY, HAVING, COUNT, WHERE
# ============================================================

query5 = """
SELECT source_ip,
       COUNT(*)                  AS high_risk_events,
       ROUND(AVG(risk_score), 2) AS avg_risk_score,
       MAX(risk_score)           AS max_risk_score
FROM ThreatEvents
WHERE risk_score >= 7.0
GROUP BY source_ip
HAVING COUNT(*) > 5
ORDER BY high_risk_events DESC, avg_risk_score DESC
LIMIT 15;
"""

result5 = pd.read_sql(query5, conn)
print(f"Task 5: Top Source IPs with >5 High-Risk Events -- {len(result5)} IPs found")
result5

Task 5: Top Source IPs with >5 High-Risk Events -- 0 IPs found


,source_ip,high_risk_events,avg_risk_score,max_risk_score


In [ ]:
# ============================================================
# TASK 6: WHERE with IN and AND - Multi-Condition Filter
# Prompt: Retrieve all TCP or SSH events that are classified
# as BruteForce or DataExfiltration with risk_score > 6.
# SQL Concepts: WHERE, IN, AND, multiple conditions
# ============================================================

query6 = """
SELECT event_id, timestamp, source_ip, destination_ip,
       protocol, label, failed_logins, bytes_sent, risk_score
FROM ThreatEvents
WHERE protocol IN ('TCP', 'SSH')
  AND label IN ('BruteForce', 'DataExfiltration')
  AND risk_score > 6.0
ORDER BY risk_score DESC;
"""

result6 = pd.read_sql(query6, conn)
print(f"Task 6: TCP/SSH BruteForce or DataExfil events with risk>6 -- {len(result6)} records")
result6.head(15)

Task 6: TCP/SSH BruteForce or DataExfil events with risk>6 -- 11 records


,event_id,timestamp,source_ip,destination_ip,protocol,label,failed_logins,bytes_sent,risk_score
0,700,2026-01-01 03:00:00,192.168.5.54,10.0.2.44,TCP,DataExfiltration,2,59795,7.06
1,97,2026-01-09 14:00:00,192.168.7.52,10.0.3.24,TCP,DataExfiltration,0,183268,6.69
2,546,2026-03-31 16:00:00,192.168.3.162,10.0.0.10,SSH,DataExfiltration,0,153915,6.65
3,237,2026-05-11 00:00:00,192.168.9.205,10.0.4.36,TCP,DataExfiltration,0,62397,6.51
4,645,2026-03-10 14:00:00,192.168.4.213,10.0.2.13,TCP,DataExfiltration,0,183873,6.42
5,72,2026-01-21 12:00:00,192.168.7.46,10.0.1.6,TCP,DataExfiltration,2,186394,6.41
6,541,2026-01-10 07:00:00,192.168.4.146,10.0.3.19,SSH,DataExfiltration,0,152547,6.22
7,842,2026-06-22 18:00:00,192.168.3.27,10.0.3.49,TCP,DataExfiltration,1,82813,6.22
8,271,2026-05-06 09:00:00,192.168.2.198,10.0.0.34,TCP,DataExfiltration,1,144154,6.21
9,27,2026-03-21 03:00:00,192.168.2.229,10.0.2.16,TCP,DataExfiltration,1,91708,6.15


In [ ]:
# ============================================================
# TASK 7: BETWEEN - Risk Score Range Analysis
# Prompt: Find all events where risk_score is between
# 5.0 and 7.5 (medium severity band) and show the count
# per label. SQL Concepts: BETWEEN, GROUP BY, COUNT
# ============================================================

query7 = """
SELECT label,
       COUNT(*)                   AS event_count,
       ROUND(MIN(risk_score), 2)  AS min_risk,
       ROUND(MAX(risk_score), 2)  AS max_risk,
       ROUND(AVG(risk_score), 2)  AS avg_risk
FROM ThreatEvents
WHERE risk_score BETWEEN 5.0 AND 7.5
GROUP BY label
ORDER BY avg_risk DESC;
"""

result7 = pd.read_sql(query7, conn)
print("Task 7: Medium Severity Events (risk_score BETWEEN 5.0 AND 7.5) by Label")
result7

Task 7: Medium Severity Events (risk_score BETWEEN 5.0 AND 7.5) by Label


,label,event_count,min_risk,max_risk,avg_risk
0,DataExfiltration,97,5.01,7.06,5.79
1,MalwareBeacon,67,5.00,6.47,5.53
2,BruteForce,55,5.00,6.22,5.36
3,SuspiciousScan,2,5.08,5.29,5.19


In [ ]:
# ============================================================
# TASK 8: LEFT JOIN - Asset-Level Threat Analysis
# Prompt: Create an Assets reference table with asset names
# and owners. Join it with ThreatEvents to show which
# named assets received high-risk events.
# SQL Concepts: CREATE TABLE, INSERT, LEFT JOIN, WHERE
# ============================================================

# Create Assets reference table
conn.execute("DROP TABLE IF EXISTS Assets")
conn.execute("""
    CREATE TABLE Assets (
        asset_ip    TEXT PRIMARY KEY,
        asset_name  TEXT,
        department  TEXT,
        owner       TEXT
    )
""")

assets_data = [
    ('10.0.0.5',  'Finance-DB',      'Finance',    'Ramesh Kumar'),
    ('10.0.0.10', 'HR-Server',       'HR',         'Priya Sharma'),
    ('10.0.0.15', 'Web-Gateway',     'IT',         'Arjun Singh'),
    ('10.0.0.20', 'Email-Server',    'IT',         'Neha Gupta'),
    ('10.0.0.25', 'Dev-Server',      'Engineering','Karan Mehta'),
    ('10.0.1.5',  'Analytics-DB',   'Data',       'Sunita Patel'),
    ('10.0.1.10', 'Backup-Server',  'IT',         'Vikram Joshi'),
    ('10.0.2.44', 'ERP-System',     'Operations', 'Anita Rao'),
]
conn.executemany("INSERT INTO Assets VALUES (?, ?, ?, ?)", assets_data)
conn.commit()

query8 = """
SELECT t.event_id,
       t.timestamp,
       t.source_ip,
       t.destination_ip,
       a.asset_name,
       a.department,
       a.owner,
       t.label,
       t.risk_score
FROM ThreatEvents t
LEFT JOIN Assets a
    ON t.destination_ip = a.asset_ip
WHERE a.asset_name IS NOT NULL
  AND t.risk_score >= 6.0
ORDER BY t.risk_score DESC
LIMIT 20;
"""

result8 = pd.read_sql(query8, conn)
print(f"Task 8: High-Risk Events on Named Assets (LEFT JOIN) -- {len(result8)} records")
result8

Task 8: High-Risk Events on Named Assets (LEFT JOIN) -- 4 records


,event_id,timestamp,source_ip,destination_ip,asset_name,department,owner,label,risk_score
0,700,2026-01-01 03:00:00,192.168.5.54,10.0.2.44,ERP-System,Operations,Anita Rao,DataExfiltration,7.06
1,546,2026-03-31 16:00:00,192.168.3.162,10.0.0.10,HR-Server,HR,Priya Sharma,DataExfiltration,6.65
2,973,2026-02-24 05:00:00,192.168.8.211,10.0.0.15,Web-Gateway,IT,Arjun Singh,DataExfiltration,6.31
3,488,2026-05-24 08:00:00,192.168.3.144,10.0.0.5,Finance-DB,Finance,Ramesh Kumar,BruteForce,6.22


In [ ]:
# ============================================================
# TASK 9: Dashboard Summary - Aggregated Threat Report
# Prompt: Generate a full dashboard summary showing per-label
# statistics: event count, total bytes sent, average risk,
# max risk, and average asset criticality.
# SQL Concepts: GROUP BY, COUNT, SUM, AVG, MAX, ROUND
# ============================================================

query9 = """
SELECT label,
       COUNT(*)                          AS total_events,
       SUM(bytes_sent)                   AS total_bytes_sent,
       ROUND(AVG(bytes_sent), 0)         AS avg_bytes_sent,
       ROUND(AVG(risk_score), 2)         AS avg_risk_score,
       MAX(risk_score)                   AS max_risk_score,
       ROUND(AVG(asset_criticality), 2)  AS avg_asset_criticality,
       SUM(failed_logins)                AS total_failed_logins
FROM ThreatEvents
GROUP BY label
ORDER BY avg_risk_score DESC;
"""

result9 = pd.read_sql(query9, conn)
print("Task 9: Full Dashboard Summary - Aggregated Threat Report by Label")
result9

Task 9: Full Dashboard Summary - Aggregated Threat Report by Label


,label,total_events,total_bytes_sent,avg_bytes_sent,avg_risk_score,max_risk_score,avg_asset_criticality,total_failed_logins
0,DataExfiltration,118,14845930,125813.0,5.60,7.06,2.96,119
1,MalwareBeacon,104,743680,7151.0,5.19,6.47,2.89,108
2,BruteForce,132,1007087,7629.0,4.82,6.22,3.15,3149
3,SuspiciousScan,73,571293,7826.0,4.07,5.29,3.26,86
4,Normal,573,4093673,7144.0,2.76,4.29,2.89,589


In [ ]:
# ============================================================
# TASK 10: Final Query - Top 10 Most Recent Critical Events
# Prompt: Find the 10 most recent events that are NOT Normal,
# have risk_score >= 5.5 and asset_criticality >= 4.
# These are the priority events for SOC analyst investigation.
# SQL Concepts: WHERE, !=, AND, ORDER BY DESC, LIMIT
# ============================================================

query10 = """
SELECT event_id,
       timestamp,
       source_ip,
       destination_ip,
       protocol,
       label,
       asset_criticality,
       failed_logins,
       bytes_sent,
       anomaly_score,
       risk_score
FROM ThreatEvents
WHERE label != 'Normal'
  AND risk_score >= 5.5
  AND asset_criticality >= 4
ORDER BY timestamp DESC, risk_score DESC
LIMIT 10;
"""

result10 = pd.read_sql(query10, conn)
total_q = """
    SELECT COUNT(*) as cnt FROM ThreatEvents
    WHERE label != 'Normal' AND risk_score >= 5.5 AND asset_criticality >= 4
"""
total_count = pd.read_sql(total_q, conn)['cnt'][0]
print("Task 10: Top 10 Most Recent Critical Events for SOC Analyst Investigation")
print(f"Total priority events matching criteria: {total_count}")
result10

Task 10: Top 10 Most Recent Critical Events for SOC Analyst Investigation
Total priority events matching criteria: 80


,event_id,timestamp,source_ip,destination_ip,protocol,label,asset_criticality,failed_logins,bytes_sent,anomaly_score,risk_score
0,936,2026-06-28 05:00:00,192.168.6.169,10.0.4.21,HTTPS,DataExfiltration,4,0,59834,2.55,5.66
1,204,2026-06-24 23:00:00,192.168.8.60,10.0.0.29,TCP,MalwareBeacon,5,1,13448,5.99,6.10
2,699,2026-06-23 18:00:00,192.168.5.230,10.0.2.37,HTTP,MalwareBeacon,4,2,6839,7.92,5.59
3,597,2026-06-23 12:00:00,192.168.7.34,10.0.3.21,UDP,DataExfiltration,4,0,169502,5.92,5.99
4,15,2026-06-21 13:00:00,192.168.4.239,10.0.1.15,TCP,MalwareBeacon,5,0,3318,4.27,5.53
5,146,2026-06-20 17:00:00,192.168.6.188,10.0.2.30,UDP,MalwareBeacon,4,0,2786,9.92,6.19
6,124,2026-06-17 21:00:00,192.168.9.207,10.0.2.11,HTTP,BruteForce,5,10,2790,6.82,5.58
7,268,2026-06-17 06:00:00,192.168.7.53,10.0.3.12,TCP,DataExfiltration,5,1,90477,3.37,5.64
8,378,2026-06-15 09:00:00,192.168.9.213,10.0.1.26,SSH,BruteForce,5,16,1828,6.80,5.78
9,568,2026-06-15 02:00:00,192.168.7.130,10.0.1.18,UDP,DataExfiltration,4,0,115792,6.03,6.40
